# Notebook 1 — Qiskit Fundamentals: Gates, Circuits, States & Visualization

**Qiskit Fall Fest 2026 — University of Ottawa**

**Difficulty:** Beginner
**Estimated time:** 100–120 minutes
**Prerequisites:** Basic Python (variables, functions, lists). No prior quantum computing or Qiskit experience required.

## Learning objectives
By the end of this notebook you will be able to:

- ✓ explain what a qubit, amplitude, and superposition are
- ✓ create and inspect quantum circuits in Qiskit
- ✓ apply single-, two-, and multi-qubit gates
- ✓ build parameterized circuits
- ✓ visualize circuits and quantum states using every major Qiskit tool
- ✓ correctly interpret Qiskit's bit ordering
- ✓ construct Bell and GHZ states

> **Note on scope.** This notebook focuses on *building and visualizing* circuits and states. Measurement statistics, observables, and expectation values are covered in depth in **Notebook 2**.

## 1. Introduction: What Is Quantum Computing?

A classical bit is always either `0` or `1`. A **qubit** can be in a combination — a *superposition* — of both:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$

Here $\alpha$ and $\beta$ are complex numbers called **amplitudes**. They are not directly observable — what we *do* observe is a probability:

$$P(0) = |\alpha|^2 \qquad P(1) = |\beta|^2 \qquad |\alpha|^2 + |\beta|^2 = 1$$

This last equation is the **normalization condition**: probabilities must sum to 1.

A few more ideas we'll build up gradually through this notebook:

- **Superposition** — a qubit can be "partly" $|0\rangle$ and "partly" $|1\rangle$ at once.
- **Relative phase** — even when two states have identical probabilities, they can differ in a complex phase that affects how they interfere with other operations. This is invisible to a simple $Z$-basis measurement but crucial to quantum algorithms.
- **Measurement** — collapses a qubit's superposition into a definite classical outcome (0 or 1), destroying the original quantum information.
- **Entanglement** — a correlation between qubits that cannot be described by treating each qubit independently. We'll build our first entangled state (the Bell state) later in this notebook.

> **Why does this matter?** Every quantum algorithm — from Grover's search to VQE for quantum chemistry — is built entirely from the gates you'll learn in this notebook.

## 2. Environment Setup

### One-time installation (before running the notebook)

> Run the commands below in a **terminal** (Anaconda Prompt on Windows), not in a Python or notebook cell. Creating a separate environment prevents package-version conflicts with your other projects. You need Conda or Miniforge installed first.

**1. Create and activate a new environment**

```bash
conda create --name qiskitff26 python=3.12 -y
conda activate qiskitff26
```

You must run `conda activate qiskitff26` again whenever you open a new terminal to work on these notebooks.

**2. Install Qiskit and all packages needed by this notebook**

```bash
python -m pip install --upgrade pip
python -m pip install "qiskit[visualization]" qiskit-aer jupyterlab ipykernel
```

The `visualization` extra installs the plotting dependencies used for circuit, Bloch-sphere, QSphere, state-city, Hinton, and Pauli-vector figures. `qiskit-aer` provides the local simulator used later in the notebook series.

**3. Make the environment available as a Jupyter kernel**

```bash
python -m ipykernel install --user --name qiskitff26 --display-name "Python (qiskitff26)"
jupyter lab
```

In Jupyter, select **Kernel → Change Kernel → Python (qiskitff26)**. Then run the version-check cell below. If that cell runs without an import error, the environment is ready.

> **Already installed?** If you already have a working Qiskit environment, simply select its kernel and continue; do not reinstall packages every time you open the notebook.

In [ ]:
import qiskit
import qiskit_aer

print("Qiskit version:     ", qiskit.__version__)
print("Qiskit Aer version: ", qiskit_aer.__version__)

- **`qiskit`** — the core SDK: circuits, gates, transpiler, quantum-info tools (`Statevector`, `SparsePauliOp`), and visualization.
- **`qiskit_aer`** — IBM's high-performance local simulator package (`AerSimulator`), used for ideal and noisy simulation (see Notebook 3).

We'll import the remaining pieces (drawing tools, `Statevector`, etc.) as we need them, so it's clear exactly which Qiskit module each tool comes from.

## 3. Creating a Quantum Circuit

In [ ]:
from qiskit import QuantumCircuit

qc_1q = QuantumCircuit(1)      # 1 qubit, no classical bits
qc_2q = QuantumCircuit(2)      # 2 qubits, no classical bits
qc_full = QuantumCircuit(2, 2) # 2 qubits AND 2 classical bits (needed to store measurement outcomes)

print("1-qubit circuit qubits:", qc_1q.num_qubits)
print("2-qubit circuit qubits:", qc_2q.num_qubits)
print("Full circuit qubits / clbits:", qc_full.num_qubits, "/", qc_full.num_clbits)

A `QuantumCircuit` is built from two kinds of registers:

- **Quantum register** — holds the qubits your circuit acts on. Each qubit starts in $|0\rangle$.
- **Classical register** — ordinary classical bits used to *store measurement results*. You only need these when you plan to measure.

Each qubit and classical bit has an **index** starting at 0 (`q0, q1, ...` and `c0, c1, ...`). A circuit is really just an ordered list of **instructions** (gates, measurements, resets, barriers) applied to specific qubit/clbit indices.

In [ ]:
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

print("num_qubits:", qc.num_qubits)
print("num_clbits:", qc.num_clbits)
print("depth():   ", qc.depth())
print("size():    ", qc.size())
print("count_ops():", qc.count_ops())

- **`depth()`** — the number of sequential "layers" of gates (roughly, how long the circuit takes to run).
- **`size()`** — the total number of instructions.
- **`count_ops()`** — a breakdown of how many times each gate type appears.

> **Try it yourself.** Add another `qc.h(1)` before the measurement and re-run. How do `depth()`, `size()`, and `count_ops()` change?

## 4. Important Quantum Gates

### 4.1 Identity ($I$)

$$I = \begin{pmatrix}1 & 0\\ 0 & 1\end{pmatrix}$$

Does nothing to the state. It's mainly useful as a placeholder or in multi-qubit Pauli strings (Notebook 2).

In [ ]:
qc = QuantumCircuit(1)
qc.id(0)
qc.draw("mpl")

### 4.2 Pauli Gates: X, Y, Z

$$X = \begin{pmatrix}0 & 1\\ 1 & 0\end{pmatrix} \qquad
Y = \begin{pmatrix}0 & -i\\ i & 0\end{pmatrix} \qquad
Z = \begin{pmatrix}1 & 0\\ 0 & -1\end{pmatrix}$$

- **X** is the quantum "bit flip": $X|0\rangle = |1\rangle$, $X|1\rangle = |0\rangle$.
- **Z** is a **phase flip**: it leaves $|0\rangle$ and $|1\rangle$ unchanged in probability but flips the sign (phase) of $|1\rangle$: $Z|1\rangle = -|1\rangle$. A $Z$-basis measurement right after $Z$ alone would show *no visible difference* from doing nothing — the effect only shows up when combined with other gates (e.g. after an $H$).
- **Y** combines both a bit flip and a phase flip.

Each Pauli gate corresponds to a 180° rotation of the Bloch sphere about its respective axis.

> **Common mistake.** Students often assume $Z$ changes measurement probabilities in the computational basis. It does not — $Z$ only changes phase, which is invisible to a direct $Z$-basis measurement.

In [ ]:
from qiskit.quantum_info import Statevector

qc_x = QuantumCircuit(1)
qc_x.x(0)
print("X|0>  ->", Statevector(qc_x))

qc_z = QuantumCircuit(1)
qc_z.x(0)   # move to |1> first
qc_z.z(0)   # now apply Z
print("Z|1>  ->", Statevector(qc_z), " (note the sign, same |0>/|1> probabilities as |1>)")

> **Try it yourself.** Predict `Y|0>` before running the next cell. Which Pauli component changes sign?

In [ ]:
qc_y = QuantumCircuit(1)
qc_y.y(0)
print("Y|0>  ->", Statevector(qc_y))

### 4.3 Hadamard ($H$) — the Gate of Superposition

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix}1 & 1\\ 1 & -1\end{pmatrix}$$

$$H|0\rangle = |+\rangle = \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle) \qquad
H|1\rangle = |-\rangle = \tfrac{1}{\sqrt2}(|0\rangle - |1\rangle)$$

$H$ turns a definite classical-like state into an equal superposition. It is the single most important gate for creating superposition and is a building block of nearly every quantum algorithm.

> **What do you expect?** Apply $H$ to $|0\rangle$. Before running the next cell — what measurement probabilities for 0 and 1 do you expect?

In [ ]:
qc_h = QuantumCircuit(1)
qc_h.h(0)
sv = Statevector(qc_h)
print(sv)
print("Probabilities:", sv.probabilities_dict())
qc_h.draw("mpl")

### 4.4 Phase Gates: S, S†, T, T†, P($\theta$)

These gates change the *relative phase* of $|1\rangle$ without altering measurement probabilities in the $Z$ basis:

- **S** applies a $90°$ phase: $S|1\rangle = i|1\rangle$
- **S†** is the inverse of S (a $-90°$ phase)
- **T** applies a $45°$ phase: $T|1\rangle = e^{i\pi/4}|1\rangle$
- **T†** is the inverse of T
- **P($\theta$)** is the general phase gate: $P(\theta)|1\rangle = e^{i\theta}|1\rangle$ — S and T are special cases of P.

In [ ]:
import numpy as np

qc_phase = QuantumCircuit(1)
qc_phase.h(0)   # go to |+> so the phase is visible on the Bloch sphere
qc_phase.s(0)
qc_phase.t(0)
qc_phase.p(np.pi / 4, 0)
qc_phase.draw("mpl")

> **Why does this matter?** Relative phase is invisible to a direct measurement but becomes visible after interference (e.g. another Hadamard). Phase gates are essential in algorithms like the Quantum Fourier Transform.

### 4.5 Rotation Gates: RX($\theta$), RY($\theta$), RZ($\theta$)

These rotate the qubit's Bloch vector by an angle $\theta$ about the X, Y, or Z axis respectively. Unlike the fixed Pauli gates, rotation gates take a **continuous parameter** — this makes them the backbone of variational algorithms (VQE, QAOA) and quantum machine learning circuits you'll meet during the hackathon.

In [ ]:
for angle, name in [(np.pi/2, "π/2"), (np.pi, "π")]:
    qc_r = QuantumCircuit(1)
    qc_r.rx(angle, 0)
    print(f"RX({name})|0> ->", Statevector(qc_r))

In [ ]:
from qiskit.visualization import plot_bloch_multivector

qc_ry = QuantumCircuit(1)
qc_ry.ry(np.pi / 2, 0)
plot_bloch_multivector(Statevector(qc_ry))

> **Exercise.** Change `ry` to `rz` in the cell above with the same angle. Why does the Bloch vector barely move visually, even though a phase was applied? *(Hint: where does $RZ$ point on the Bloch sphere relative to $|0\rangle$?)*

### 4.6 Square-Root of X ($\sqrt{X}$ / `sx`)

$SX$ satisfies $SX \cdot SX = X$ — applying it twice equals a full X gate. You won't use it directly very often as a beginner, but it's worth recognizing: IBM hardware's native basis gate set includes `sx`, so it frequently appears automatically after **transpilation** (Notebook 4).

In [ ]:
qc_sx = QuantumCircuit(1)
qc_sx.sx(0)
qc_sx.sx(0)  # two SX = one X
print(Statevector(qc_sx))

## 5. Two-Qubit and Controlled Gates

Controlled gates act on two roles:

- the **control qubit** — its state determines *whether* the gate fires
- the **target qubit** — the qubit the gate actually acts on

**CX (CNOT)** flips the target *only if* the control is $|1\rangle$. When the control is in a classical state ($|0\rangle$ or $|1\rangle$), this looks like classical conditional logic. But **when the control is in superposition, CNOT does not "copy" a bit** — it creates *entanglement*, correlating the two qubits without giving either one a definite value on its own. We'll see this directly in the Bell-state section below.

Other common two-qubit gates:

- **CZ** — applies a phase flip to the target only if control is $|1\rangle$ (symmetric in control/target)
- **CY** — controlled-Y
- **CH** — controlled-Hadamard
- **SWAP** — exchanges the states of two qubits entirely (no control qubit — always active)

In [ ]:
qc_cx = QuantumCircuit(2)
qc_cx.x(0)      # control = |1>
qc_cx.cx(0, 1)  # target flips because control is |1>
print("Control=1 case:", Statevector(qc_cx))

qc_swap = QuantumCircuit(2)
qc_swap.x(0)
qc_swap.swap(0, 1)
print("After SWAP:    ", Statevector(qc_swap))

> **Common mistake.** "CNOT copies the control qubit onto the target" is only true when the control is a *classical* $|0\rangle$ or $|1\rangle$. When the control is in superposition, CNOT instead produces entanglement — a genuinely quantum correlation with no classical analogue.

### Controlled Rotations: CRX, CRY, CRZ, CP

These apply a rotation to the target *conditioned* on the control qubit, and take a continuous angle parameter. They are especially important in variational quantum circuits and quantum machine learning models you may use during the hackathon.

In [ ]:
qc_crot = QuantumCircuit(2)
qc_crot.h(0)
qc_crot.cry(np.pi / 3, 0, 1)
qc_crot.draw("mpl")

## 6. Multi-Qubit Gates

- **CCX / Toffoli** — flips the target only if *both* control qubits are $|1\rangle$. The quantum analogue of an AND gate.
- **CSWAP / Fredkin** — swaps two target qubits only if the control qubit is $|1\rangle$.

You don't need deep algorithmic use of these yet — just recognize them and know how to place them.

In [ ]:
qc_multi = QuantumCircuit(3)
qc_multi.x([0, 1])       # both controls = |1>
qc_multi.ccx(0, 1, 2)    # Toffoli: target flips since both controls are 1
print(Statevector(qc_multi))
qc_multi.draw("mpl")

## 7. Circuit Operations

In [ ]:
qc_ops = QuantumCircuit(2, 2)
qc_ops.h(0)
qc_ops.barrier()          # visual/scheduling separator; no effect on the state
qc_ops.cx(0, 1)
qc_ops.reset(0)           # forces qubit 0 back to |0>, discarding its current state
qc_ops.measure([0, 1], [0, 1])
qc_ops.draw("mpl")

- **`barrier()`** — a visual/scheduling boundary; does not change the quantum state, but prevents the transpiler from reordering gates across it.
- **`reset(qubit)`** — forcibly resets a qubit to $|0\rangle$, discarding any superposition or entanglement it had.
- **`measure(qubits, clbits)`** — measures specific qubits into specific classical bits.
- **`measure_all()`** — a shortcut that measures every qubit into a *new* classical register called `meas`.

In [ ]:
base = QuantumCircuit(2)
base.h(0)
base.cx(0, 1)

bell_gate = base.to_gate(label="BellPrep")   # turn a subcircuit into a reusable gate
inverse_gate = bell_gate.inverse()           # its inverse (uncompute) operation

big = QuantumCircuit(3)
big.append(bell_gate, [0, 1])
big.append(inverse_gate, [1, 2])
big.draw("mpl")

This shows **composition** in action: build a small circuit, convert it into a single reusable instruction with `to_gate()`, and `append()` it (or its `inverse()`) into a larger circuit. This pattern is common when building repeated ansatz layers in variational algorithms.

> **Try it yourself.** Use `base.copy()` to make an independent copy of `base`, modify the copy (e.g. add another `h`), and confirm the original `base` circuit is unaffected.

## 8. Parameterized Circuits

Instead of a fixed numeric angle, we can use a symbolic **`Parameter`**. This lets us build one circuit *template* and later bind in different numeric values — exactly the pattern used in variational algorithms and QML models.

In [ ]:
from qiskit.circuit import Parameter

theta = Parameter("θ")

qc_param = QuantumCircuit(1)
qc_param.ry(theta, 0)
qc_param.draw("mpl")

In [ ]:
bound_qc = qc_param.assign_parameters({theta: np.pi / 2})
print(Statevector(bound_qc))

# assign_parameters can also bind several values at once via a list, for sweeps:
sweep_values = [0, np.pi / 4, np.pi / 2, np.pi]
for v in sweep_values:
    sv = Statevector(qc_param.assign_parameters({theta: v}))
    print(f"θ={v:.3f} -> {sv}")

> **Why does this matter?** Parameterized rotation gates like `RY(θ)` are used extensively in variational quantum algorithms (VQE, QAOA) and quantum machine-learning circuits — the hackathon's QML track builds directly on this pattern.

## 9. Circuit Visualization — Read This Carefully

Qiskit supports several circuit-drawing backends via `qc.draw(output=...)`:

- **`"text"`** — ASCII-art rendering, works everywhere, no dependencies.
- **`"mpl"`** — Matplotlib rendering, the most common choice for reports and slides.
- **`"latex"`** — renders as a compiled image via LaTeX (requires a working LaTeX installation on your machine).
- **`"latex_source"`** — returns the raw LaTeX source string instead of rendering it (useful if you want to embed it in a paper and don't have LaTeX installed locally).

In [ ]:
qc_demo = QuantumCircuit(3, 3)
qc_demo.h(0)
qc_demo.cx(0, 1)
qc_demo.cx(1, 2)
qc_demo.barrier()
qc_demo.measure([0, 1, 2], [0, 1, 2])

print(qc_demo.draw("text"))

In [ ]:
qc_demo.draw("mpl")

In [ ]:
print(qc_demo.draw("latex_source"))
# qc_demo.draw("latex")  # uncomment if you have a local LaTeX installation

Useful drawing options (verified against current Qiskit documentation):

- **`reverse_bits=True`** — draws qubit 0 at the bottom instead of the top (matches some textbook conventions).
- **`plot_barriers=False`** — hides barrier markers for a cleaner look.
- **`fold=-1`** — draws the whole circuit on one line without wrapping, useful for wide circuits.
- **`idle_wires=False`** — hides qubits that have no gates applied to them.

In [ ]:
qc_demo.draw("mpl", reverse_bits=True, plot_barriers=False, fold=-1)

In [ ]:
fig = qc_demo.draw("mpl")
fig.savefig("circuit_example.png", dpi=150, bbox_inches="tight")
print("Saved circuit_example.png")

## 10. Quantum State Visualization

`Statevector` lets us inspect the *exact* mathematical state produced by a circuit (something only a simulator can give you — real hardware cannot report a statevector directly, only measurement outcomes; see Notebook 2).

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
sv_bell = Statevector(bell)
sv_bell.draw("latex")

In [ ]:
from qiskit.visualization import (
    plot_bloch_multivector,
    plot_state_qsphere,
    plot_state_city,
    plot_state_hinton,
    plot_state_paulivec,
)

plot_bloch_multivector(sv_bell)

**Bloch multivector** — plots each qubit's *individual* reduced Bloch vector. Excellent for single-qubit states. **Limitation:** for an entangled state like the Bell state, each individual Bloch vector shrinks toward the center — this is a visual signature of entanglement, but the plot cannot show *how* the qubits are correlated, only that each one looks locally mixed.

In [ ]:
plot_state_qsphere(sv_bell)

**QSphere** — a Qiskit-specific view where every possible multi-qubit outcome is a point on a sphere; dot size shows amplitude magnitude, color shows relative phase. This is the best tool for seeing *global* multi-qubit structure, including entanglement, in one picture.

In [ ]:
plot_state_city(sv_bell)

**State city** — a 3D bar chart of the real and imaginary parts of the state (or density matrix). Good for seeing exact numerical structure.

In [ ]:
plot_state_hinton(sv_bell)

**Hinton plot** — square size encodes magnitude of each density-matrix element; a quick way to visually confirm which basis states carry weight.

In [ ]:
plot_state_paulivec(sv_bell)

**Pauli vector plot** — shows the expectation value of every Pauli operator (I, X, Y, Z and their tensor products) for the state. This directly previews **Notebook 2**, where Pauli observables and expectation values become central.

> **Important.** These visualizations are *not* interchangeable "pretty pictures" — each answers a different question:
> - Bloch multivector → what does *each individual qubit* look like on average?
> - QSphere → what is the *global* amplitude/phase structure, including entanglement signatures?
> - City / Hinton → what are the exact numeric (real/imaginary or magnitude) components?
> - Pauli vector → what are the expectation values of Pauli observables (ties directly into Notebook 2)?
>
> **Single-qubit Bloch vectors do not fully describe entangled multi-qubit states** — this is why the Bloch multivector plot looks "shrunk" for the Bell state.

## 11. Qiskit Bit Ordering — Read This Before the Hackathon

This is one of the most common sources of confusion for Qiskit beginners.

Qiskit uses **little-endian** ordering: in a bitstring like `"01"`, the **rightmost** character is qubit 0, and the **leftmost** character is the highest-indexed qubit.

**Example 1.** For a 2-qubit circuit, the bitstring `"01"` means: qubit 1 = 0, qubit 0 = 1 (read right to left).

**Example 2.** If you apply `qc.x(0)` only (flip qubit 0, leave qubit 1 alone) and measure both, you get `"01"` — *not* `"10"` — because qubit 0 (now `1`) is on the right.

**Example 3.** In a 3-qubit circuit, the bitstring `"100"` means qubit 2 = 1, qubit 1 = 0, qubit 0 = 0.

This convention affects: how you read `plot_histogram` labels, how `Statevector` basis labels are ordered, and how circuit diagrams (which draw qubit 0 at the *top* by default) relate to the bitstrings you get back. Keep this firmly in mind during the hackathon when interpreting `counts` dictionaries.

In [ ]:
qc_bits = QuantumCircuit(3, 3)
qc_bits.x(0)  # flip only qubit 0
qc_bits.measure([0, 1, 2], [0, 1, 2])
print("Circuit with qubit 0 flipped, followed by measurement:")
qc_bits.draw("mpl")

In [ ]:
only_x0 = QuantumCircuit(3)
only_x0.x(0)
print("Statevector label for 'qubit 0 flipped only':", Statevector(only_x0))
# Note the ket label is |001>, i.e. the rightmost character corresponds to qubit 0.

## 12. The Bell State

The canonical entangled 2-qubit state:

$$|00\rangle \xrightarrow{H \text{ on } q_0} \tfrac{1}{\sqrt2}(|00\rangle + |10\rangle) \xrightarrow{CX(0,1)} \tfrac{1}{\sqrt2}(|00\rangle + |11\rangle)$$

> **Checkpoint.** Without running any code: after `H` on qubit 0 alone, what are $P(0)$ and $P(1)$ for qubit 0? What is qubit 1's state at that point?

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.draw("mpl")

In [ ]:
sv_bell = Statevector(bell)
sv_bell.draw("latex")

In [ ]:
plot_state_qsphere(sv_bell)

In [ ]:
plot_state_city(sv_bell)

In [ ]:
plot_bloch_multivector(sv_bell)

**Interpretation.** The state is $\tfrac{1}{\sqrt2}(|00\rangle + |11\rangle)$ — only the "both zero" and "both one" outcomes have nonzero amplitude. Notice the Bloch multivector shows both qubits shrunk toward the center of their spheres: individually, each qubit looks maximally uncertain, yet the *pair* is perfectly correlated. This is entanglement — information that only exists in the *joint* description of both qubits, not in either one alone. Notebook 2 will make this precise using $\langle ZZ\rangle$ and $\langle XX\rangle$ expectation values.

## 13. The GHZ State

The GHZ state generalizes the Bell state to $n$ qubits:

$$|GHZ_n\rangle = \tfrac{1}{\sqrt2}(|0\rangle^{\otimes n} + |1\rangle^{\otimes n})$$

In [ ]:
ghz3 = QuantumCircuit(3)
ghz3.h(0)
ghz3.cx(0, 1)
ghz3.cx(1, 2)
ghz3.draw("mpl")

In [ ]:
print(Statevector(ghz3))
plot_state_qsphere(Statevector(ghz3))

In [ ]:
def ghz_circuit(n: int) -> QuantumCircuit:
    """Return an n-qubit GHZ-state preparation circuit."""
    qc = QuantumCircuit(n)
    qc.h(0)
    for q in range(n - 1):
        qc.cx(q, q + 1)
    return qc

ghz5 = ghz_circuit(5)
ghz5.draw("mpl")

## 14. Exercises

Work through these before the hackathon. Solutions are in the final collapsed section — try each exercise yourself first.

**Exercise 1 — Try it yourself.** Prepare $|1\rangle$ from $|0\rangle$ using a single gate.

**Exercise 2 — Try it yourself.** Prepare $|+\rangle$.

**Exercise 3 — Exercise.** Prepare $|-\rangle$. *(Hint: apply X before H, or H then Z — verify both give the same state.)*

**Exercise 4 — Exercise.** Apply `RX(π)` to $|0\rangle$. Predict the resulting state (up to global phase) before running your code, then check with `Statevector`.

**Exercise 5 — Exercise.** Build a Bell state using `cx` and `h` in a **different qubit order** than Section 12 (i.e. put the Hadamard on qubit 1 and control the CX from qubit 1). Is the resulting state the same?

**Exercise 6 — Challenge.** Build a 3-qubit GHZ state *without* using the `ghz_circuit` function — write it out gate by gate — and confirm it matches `ghz_circuit(3)`.

**Exercise 7 — Exercise.** Create a parameterized circuit with a single `RY(θ)` gate and bind three different angles, printing the `Statevector` for each.

**Exercise 8 — Challenge.** Reproduce the circuit drawn below using gates you've learned in this notebook (don't peek at the code that generated it until you've tried):

In [ ]:
_hidden_challenge = QuantumCircuit(2)
_hidden_challenge.h(0)
_hidden_challenge.x(1)
_hidden_challenge.cx(0, 1)
_hidden_challenge.z(1)
_hidden_challenge.draw("mpl")

### Solutions (collapsed — try the exercises first!)

```python
# Exercise 1
qc1 = QuantumCircuit(1); qc1.x(0)

# Exercise 2
qc2 = QuantumCircuit(1); qc2.h(0)

# Exercise 3
qc3 = QuantumCircuit(1); qc3.x(0); qc3.h(0)

# Exercise 4
qc4 = QuantumCircuit(1); qc4.rx(np.pi, 0)
print(Statevector(qc4))  # ~ -i|1>, same probabilities as X|0>

# Exercise 5
qc5 = QuantumCircuit(2); qc5.h(1); qc5.cx(1, 0)
print(Statevector(qc5))  # same Bell state, up to qubit labeling

# Exercise 6
qc6 = QuantumCircuit(3); qc6.h(0); qc6.cx(0, 1); qc6.cx(1, 2)

# Exercise 7
th = Parameter("θ")
qc7 = QuantumCircuit(1); qc7.ry(th, 0)
for v in [0, np.pi/3, np.pi]:
    print(Statevector(qc7.assign_parameters({th: v})))

# Exercise 8 — matches _hidden_challenge above
qc8 = QuantumCircuit(2)
qc8.h(0); qc8.x(1); qc8.cx(0, 1); qc8.z(1)
```

## Qiskit Cheat Sheet — Notebook 1

| Task | Code |
|---|---|
| Create a circuit | `QuantumCircuit(n_qubits, n_clbits)` |
| Pauli gates | `qc.x(q)`, `qc.y(q)`, `qc.z(q)` |
| Hadamard | `qc.h(q)` |
| Phase gates | `qc.s(q)`, `qc.sdg(q)`, `qc.t(q)`, `qc.tdg(q)`, `qc.p(theta, q)` |
| Rotations | `qc.rx(theta, q)`, `qc.ry(theta, q)`, `qc.rz(theta, q)` |
| Sqrt-X | `qc.sx(q)` |
| Two-qubit gates | `qc.cx(c, t)`, `qc.cz(c, t)`, `qc.cy(c, t)`, `qc.ch(c, t)`, `qc.swap(a, b)` |
| Controlled rotations | `qc.crx/cry/crz/cp(theta, c, t)` |
| Multi-qubit gates | `qc.ccx(c1, c2, t)`, `qc.cswap(c, a, b)` |
| Barrier / reset | `qc.barrier()`, `qc.reset(q)` |
| Measurement | `qc.measure(q, c)`, `qc.measure_all()` |
| Parameters | `Parameter("θ")`, `qc.assign_parameters({theta: value})` |
| Draw | `qc.draw("text"/"mpl"/"latex"/"latex_source")` |
| Exact state | `Statevector(qc)` |
| State plots | `plot_bloch_multivector`, `plot_state_qsphere`, `plot_state_city`, `plot_state_hinton`, `plot_state_paulivec` |
| Inspect circuit | `qc.num_qubits`, `qc.depth()`, `qc.size()`, `qc.count_ops()` |
| Compose | `qc.to_gate()`, `qc.append(gate, qubits)`, `gate.inverse()`, `qc.copy()` |

## Common Mistakes

- **Confusing qubit order in bitstrings.** Qiskit is little-endian — qubit 0 is the *rightmost* character. Re-read Section 11 if unsure.
- **Assuming Z changes measurement probabilities.** It only changes phase; probabilities in the Z-basis are unaffected.
- **Treating CNOT as "copying" a qubit.** True only when the control is classical (definite $|0\rangle$/$|1\rangle$); in superposition, CNOT creates entanglement instead.
- **Forgetting a classical register when you plan to measure.** `QuantumCircuit(n)` alone has no classical bits — use `QuantumCircuit(n, n)` or `measure_all()`.
- **Confusing Bloch multivector "shrinkage" with a bug.** A shrunk individual Bloch vector on an entangled qubit is expected behaviour, not an error.
- **Using deprecated patterns from old tutorials**, e.g. `execute(qc, backend)`, `Aer.get_backend(...)`, or `qc.draw()` without an explicit backend string in newer contexts — this notebook and the rest of the series consistently use the current Qiskit 2.x / Aer 0.17.x / Runtime 0.47.x APIs.